# Solving CartPole Without Gradients: Simulated Annealing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/simulated_annealing_cartpole.ipynb)

This notebook implements **hill climbing with annealing step size** to solve CartPole-v1 using just 4 parameters. The algorithm:

1. Maintains a single set of policy parameters
2. Perturbs them with uniform noise scaled by a step size `alpha`
3. Accepts the perturbation only if it improves the score
4. Shrinks `alpha` by a factor of 0.9 on each improvement

No gradients, no population, no neural network. Just four numbers and a cooling schedule.

**Blog post:** [sesen.ai/blog/simulated-annealing-cartpole](https://sesen.ai/blog/simulated-annealing-cartpole)

---

## Setup

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

## Core Functions

Our policy is a simple linear classifier: push right if `theta @ obs > 0`, push left otherwise. We evaluate each candidate over 10 episodes to reduce noise from stochastic initial conditions.

In [ ]:
def evaluate_policy(env_name, theta, n_episodes=10):
    """Run multiple episodes with a linear policy and return the average reward."""
    total_reward = 0
    for _ in range(n_episodes):
        env = gym.make(env_name)
        obs, _ = env.reset()
        episode_reward = 0
        done = False
        while not done:
            action = 1 if np.dot(theta, obs) > 0 else 0
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        env.close()
        total_reward += episode_reward
    return total_reward / n_episodes

In [ ]:
def simulated_annealing(env_name, n_params, n_iter=80, n_eval_episodes=10,
                        alpha=1.0, decay=0.9):
    """Hill climbing with annealing step size for policy search.

    Hyperparameters preserved from original code:
    - alpha=1.0 initial step size
    - decay=0.9 (alpha *= 0.9 on improvement)
    - 10 episodes per evaluation
    - 80 iterations
    - Uniform perturbation in [-0.5*alpha, 0.5*alpha]
    """
    best_theta = np.zeros(n_params)
    best_score = evaluate_policy(env_name, best_theta, n_eval_episodes)

    history = {
        'best_scores': [best_score],
        'candidate_scores': [best_score],
        'alphas': [alpha],
        'accepted': [True],
    }

    for i in range(n_iter):
        # Perturb current best (uniform noise scaled by alpha)
        perturbation = (np.random.rand(n_params) - 0.5) * alpha
        candidate = best_theta + perturbation

        # Evaluate candidate over multiple episodes
        score = evaluate_policy(env_name, candidate, n_eval_episodes)

        # Accept only if strictly better, then shrink step size
        accepted = False
        if score > best_score:
            best_theta = candidate
            best_score = score
            alpha *= decay
            accepted = True

        history['best_scores'].append(best_score)
        history['candidate_scores'].append(score)
        history['alphas'].append(alpha)
        history['accepted'].append(accepted)

        marker = '\u2713' if accepted else '\u2717'
        print(f"Iter {i+1:3d} | Candidate: {score:6.1f} | Best: {best_score:6.1f} | Alpha: {alpha:.4f} | {marker}")

    return best_theta, best_score, history

## Run Simulated Annealing

Let's solve CartPole-v1 (max 500 steps per episode). With the original hyperparameters:

In [ ]:
np.random.seed(42)

best_theta, best_score, history = simulated_annealing(
    'CartPole-v1', n_params=4, n_iter=80, n_eval_episodes=10,
    alpha=1.0, decay=0.9
)

print(f"\nBest theta: {best_theta}")
print(f"Best average score: {best_score}")
print(f"Improvements found: {sum(history['accepted'][1:])} / 80")
print(f"Final alpha: {history['alphas'][-1]:.4f}")

### Final Evaluation

Verify the solution with 100 independent episodes:

In [ ]:
final_scores = [evaluate_policy('CartPole-v1', best_theta, n_episodes=1) for _ in range(100)]
print(f"Final evaluation (100 episodes): {np.mean(final_scores):.0f} +/- {np.std(final_scores):.0f}")

## Visualisation: Training Curve

The staircase pattern shows accepted improvements (green) vs rejected candidates (red). The dashed grey line tracks the step size alpha on the secondary axis.

In [ ]:
iters = list(range(len(history['best_scores'])))

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.scatter(iters[1:], history['candidate_scores'][1:],
            c=['#2ecc71' if a else '#e74c3c' for a in history['accepted'][1:]],
            alpha=0.5, s=20, zorder=2)
ax1.plot(iters, history['best_scores'], 'b-', linewidth=2, zorder=3, label='Best score')
ax1.axhline(y=500, color='k', linestyle=':', alpha=0.3, label='Max possible (500)')
ax1.set_xlabel('Iteration', fontsize=11)
ax1.set_ylabel('Average Reward (10 episodes)', fontsize=11)
ax1.set_ylim(-10, 530)
ax1.legend(loc='center right', fontsize=9)

ax2 = ax1.twinx()
ax2.plot(iters, history['alphas'], 'k--', alpha=0.4, linewidth=1, label='Step size (\u03b1)')
ax2.set_ylabel('Step size (\u03b1)', fontsize=11, color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.set_ylim(-0.05, 1.15)
ax2.legend(loc='upper right', fontsize=9)

fig.tight_layout()
plt.show()

## Visualisation: Convergence Animation

In [ ]:
n_frames = 15
frame_indices = np.linspace(0, len(history['best_scores'])-1, n_frames, dtype=int)

fig, ax = plt.subplots(figsize=(8, 4))

def update(frame_num):
    ax.clear()
    idx = frame_indices[frame_num]
    x = list(range(idx + 1))
    y = history['best_scores'][:idx + 1]
    ax.plot(x, y, 'b-', linewidth=2, label='Best score')
    if idx > 0:
        cand_x = list(range(1, idx + 1))
        cand_y = history['candidate_scores'][1:idx + 1]
        colors = ['#2ecc71' if a else '#e74c3c' for a in history['accepted'][1:idx + 1]]
        ax.scatter(cand_x, cand_y, c=colors, alpha=0.5, s=20, zorder=2)
    ax.axhline(y=500, color='k', linestyle=':', alpha=0.3)
    ax.set_xlim(-2, 82)
    ax.set_ylim(-10, 530)
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Average Reward', fontsize=11)
    ax.set_title(f'Iteration {idx}  |  Best: {y[-1]:.0f}  |  \u03b1: {history["alphas"][idx]:.3f}', fontsize=11)
    ax.legend(loc='center right', fontsize=9)
    return []

anim = FuncAnimation(fig, update, frames=n_frames, interval=500, blit=True)
anim.save('/tmp/sa_convergence.gif', writer=PillowWriter(fps=2), dpi=100)
plt.close(fig)
display(Image(filename='/tmp/sa_convergence.gif'))

## Visualisation: Cooling Schedule

The step size only decays when an improvement is found (green vertical bands). Compare with fixed geometric decay schedules.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: alpha over iterations
ax1.plot(iters, history['alphas'], 'b-', linewidth=2)
ax1.set_xlabel('Iteration', fontsize=11)
ax1.set_ylabel('Step size (\u03b1)', fontsize=11)
ax1.set_title('Step Size Over Time', fontsize=12)
for i in range(1, len(history['accepted'])):
    if history['accepted'][i]:
        ax1.axvline(x=i, color='green', alpha=0.15)

# Right: geometric decay comparison
steps = np.arange(30)
schedules = {
    'Our schedule (\u03b1 \u00d7 0.9)': 1.0 * 0.9**steps,
    'Faster (\u03b1 \u00d7 0.8)': 1.0 * 0.8**steps,
    'Slower (\u03b1 \u00d7 0.95)': 1.0 * 0.95**steps,
}
for label, vals in schedules.items():
    ax2.plot(steps, vals, linewidth=2, label=label)
ax2.set_xlabel('Number of Improvements', fontsize=11)
ax2.set_ylabel('Step size (\u03b1)', fontsize=11)
ax2.set_title('Geometric Decay Schedules', fontsize=12)
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

## Comparison: SA vs Random Search

How much does building on previous improvements help? Random search samples a fresh random policy each iteration.

In [ ]:
def random_search(env_name, n_params, n_iter=80, n_eval_episodes=10):
    """Random search baseline: sample uniformly each iteration."""
    best_theta = np.zeros(n_params)
    best_score = evaluate_policy(env_name, best_theta, n_eval_episodes)
    history = {'best_scores': [best_score], 'candidate_scores': [best_score]}

    for i in range(n_iter):
        candidate = (np.random.rand(n_params) - 0.5) * 2
        score = evaluate_policy(env_name, candidate, n_eval_episodes)
        if score > best_score:
            best_theta = candidate
            best_score = score
        history['best_scores'].append(best_score)
        history['candidate_scores'].append(score)
    return best_theta, best_score, history

np.random.seed(123)
_, rs_best, rs_hist = random_search('CartPole-v1', n_params=4)
print(f"Random search best: {rs_best:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
iters_plot = list(range(len(history['best_scores'])))
ax.plot(iters_plot, history['best_scores'], 'b-', linewidth=2, label='Simulated annealing (best)')
ax.plot(iters_plot, rs_hist['best_scores'], 'r--', linewidth=2, label='Random search (best)')
ax.axhline(y=500, color='k', linestyle=':', alpha=0.3, label='Max possible (500)')
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Best Average Reward', fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(-10, 530)
fig.tight_layout()
plt.show()

## Exercises

### 1. Decay Rate Sweep

Try `decay` values of 0.8, 0.9, 0.95, and 0.99. How does the cooling speed affect convergence? Is there a sweet spot?

In [ ]:
# Exercise 1: Uncomment and run
# decay_values = [0.8, 0.9, 0.95, 0.99]
# for d in decay_values:
#     np.random.seed(42)
#     _, score, _ = simulated_annealing('CartPole-v1', 4, decay=d)
#     print(f"decay={d}: best score = {score:.1f}")

### 2. True Simulated Annealing

Modify the algorithm to accept worse solutions with probability `exp(-delta/T)` where `delta` is the score difference and `T` decays on a fixed schedule. Does it help on CartPole?

In [ ]:
# Exercise 2: Implement true SA
# def true_simulated_annealing(env_name, n_params, n_iter=80, n_eval_episodes=10,
#                               T_start=100.0, T_end=0.1):
#     best_theta = np.zeros(n_params)
#     current_theta = best_theta.copy()
#     current_score = evaluate_policy(env_name, current_theta, n_eval_episodes)
#     best_score = current_score
#
#     for i in range(n_iter):
#         T = T_start * (T_end / T_start) ** (i / n_iter)  # Exponential cooling
#         perturbation = (np.random.rand(n_params) - 0.5)
#         candidate = current_theta + perturbation
#         score = evaluate_policy(env_name, candidate, n_eval_episodes)
#
#         delta = score - current_score
#         if delta > 0 or np.random.rand() < np.exp(delta / T):
#             current_theta = candidate
#             current_score = score
#             if score > best_score:
#                 best_theta = candidate.copy()
#                 best_score = score
#
#     return best_theta, best_score

### 3. Seed Sensitivity

Run the algorithm 20 times with different seeds. What fraction of runs reach 500? How does this compare to CEM's reliability?

In [ ]:
# Exercise 3: Uncomment and run (takes ~5 minutes)
# results = []
# for seed in range(20):
#     np.random.seed(seed)
#     _, score, _ = simulated_annealing('CartPole-v1', 4, n_iter=80)
#     results.append(score)
#     print(f"Seed {seed:2d}: {score:.0f}")
# print(f"\nReached 500: {sum(r >= 500 for r in results)}/20")
# print(f"Mean: {np.mean(results):.0f}, Min: {np.min(results):.0f}")

### 4. Harder Environments

Try SA on `Acrobot-v1` or `MountainCar-v0`. Does the 4-parameter linear policy have enough capacity?

In [ ]:
# Exercise 4: Try different environments
# For Acrobot-v1 (6-dimensional observation, 3 actions):
# def evaluate_acrobot(theta, n_episodes=10):
#     total = 0
#     for _ in range(n_episodes):
#         env = gym.make('Acrobot-v1')
#         obs, _ = env.reset()
#         reward_sum = 0
#         done = False
#         while not done:
#             scores = [np.dot(theta[i*6:(i+1)*6], obs) for i in range(3)]
#             action = np.argmax(scores)
#             obs, reward, terminated, truncated, _ = env.step(action)
#             reward_sum += reward
#             done = terminated or truncated
#         env.close()
#         total += reward_sum
#     return total / n_episodes

---

**Further reading:**
- [Blog post](https://sesen.ai/blog/simulated-annealing-cartpole) for the full tutorial with theory and paper references
- [Kirkpatrick, Gelatt, and Vecchi (1983)](https://doi.org/10.1126/science.220.4598.671) — the foundational SA paper
- [Cross-Entropy Method notebook](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/cross_entropy_method.ipynb) — the population-based companion